In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import json
import os

In [ ]:
df=pd.read_csv("../../data/raw/raw.csv")

In [ ]:
df.info()

In [ ]:
df.drop(columns=['description','action_taken_timestamp','closed_datetime'],inplace=True)

In [ ]:
df['created_datetime'] = pd.to_datetime(df['created_datetime'], errors='coerce')
df['modified_datetime'] = pd.to_datetime(df['modified_datetime'], errors='coerce')

In [ ]:
df = df.dropna(subset=['created_datetime','modified_datetime'])

In [ ]:
len(df)

In [ ]:
df['created_datetime'] = df['created_datetime'].dt.tz_convert('Asia/Kolkata')
df['modified_datetime'] = df['modified_datetime'].dt.tz_convert('Asia/Kolkata')

In [ ]:
df['cdt_minute of day'] = (df['created_datetime'].dt.hour * 60) + df['created_datetime'].dt.minute
df['cdt_hour'] = df['created_datetime'].dt.hour
df['cdt_day_of_week'] = df['created_datetime'].dt.day_name()
df['cdt_month'] = df['created_datetime'].dt.month
df['cdt_date'] = df['created_datetime'].dt.date

In [ ]:
df['mdt_minute of day'] = (df['modified_datetime'].dt.hour * 60) + df['modified_datetime'].dt.minute
df['mdt_hour'] = df['modified_datetime'].dt.hour
df['mdt_day_of_week'] = df['modified_datetime'].dt.day_name()
df['mdt_month'] = df['modified_datetime'].dt.month
df['mdt_date'] = df['modified_datetime'].dt.date

In [ ]:
df['processing_duration_minutes'] = (df['modified_datetime'] - df['created_datetime']).dt.total_seconds() / 60

In [ ]:
df.sample(5)

In [ ]:
sns.boxplot(df['processing_duration_minutes'])

In [ ]:
df['processing_duration_minutes'].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99])

In [ ]:
OUT_DIR = "../../data/processed"

PARKING_TYPES = {
    "NO PARKING", "WRONG PARKING", "DOUBLE PARKING", "PARKING IN A MAIN ROAD",
    "PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC", "PARKING NEAR ROAD CROSSING",
    "PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS", "PARKING ON FOOTPATH",
    "PARKING OPPOSITE TO ANOTHER PARKED VEHICLE", "PARKING OTHER THAN BUS STOP",
}

VEHICLE_FOOTPRINT_WEIGHT = {
    "SCOOTER": 1.0, "MOTOR CYCLE": 1.0, "MOPED": 1.0,
    "CAR": 2.5, "JEEP": 2.5, "VAN": 2.5, "PASSENGER AUTO": 2.0,
    "GOODS AUTO": 2.0, "TEMPO": 3.0, "LGV": 3.5, "MAXI-CAB": 3.0,
    "PRIVATE BUS": 5.0, "BUS (BMTC/KSRTC)": 5.0, "TOURIST BUS": 5.0,
    "FACTORY BUS": 5.0, "SCHOOL VEHICLE": 5.0, "HGV": 5.5, 
    "LORRY/GOODS VEHICLE": 5.5, "MINI LORRY": 4.0, "TRACTOR": 4.0, 
    "TANKER": 5.5, "OTHERS": 2.0,
}

In [ ]:
def clean_pipeline(dataframe: pd.DataFrame):
    df = dataframe.copy()
    
    # 1. Parse Datetimes
    df["created_datetime"] = pd.to_datetime(df["created_datetime"], errors='coerce')
    df["modified_datetime"] = pd.to_datetime(df["modified_datetime"], errors='coerce')
    df = df.dropna(subset=["created_datetime", "modified_datetime"]).copy()
    df["created_datetime"] = df["created_datetime"].dt.tz_convert("Asia/Kolkata")
    df["modified_datetime"] = df["modified_datetime"].dt.tz_convert("Asia/Kolkata")
    
    # 2. Parse JSON Lists
    def safe_json(x):
        try:
            return json.loads(x)
        except (TypeError, json.JSONDecodeError):
            return []
    df["violation_type_list"] = df["violation_type"].apply(safe_json)
    df["offence_code_list"] = df["offence_code"].apply(safe_json)
    df["n_violations"] = df["violation_type_list"].apply(len)
    df["has_parking_violation"] = df["violation_type_list"].apply(
        lambda lst: any(v in PARKING_TYPES for v in lst)
    )
    df["is_pure_parking"] = df["violation_type_list"].apply(
        lambda lst: len(lst) > 0 and all(v in PARKING_TYPES for v in lst)
    )
    
    # 3. Fix Processing Durations
    df["processing_duration_minutes"] = (df["modified_datetime"] - df["created_datetime"]).dt.total_seconds() / 60.0
    df = df.rename(columns={"processing_duration_minutes": "record_lifecycle_minutes_RAW_UNRELIABLE"})
    raw = df["record_lifecycle_minutes_RAW_UNRELIABLE"]
    p95 = raw.quantile(0.95)
    df["record_lifecycle_minutes_capped_p95"] = raw.clip(lower=0, upper=p95)
    df["record_lifecycle_is_negative_artifact"] = raw < 0
    
    # 4. Clean Categoricals
    df["vehicle_type"] = df["vehicle_type"].str.strip().str.upper()
    df["vehicle_footprint_weight"] = df["vehicle_type"].map(VEHICLE_FOOTPRINT_WEIGHT).fillna(2.0)
    df["junction_name"] = df["junction_name"].fillna("No Junction").str.strip()
    df["has_named_junction"] = df["junction_name"] != "No Junction"
    df["police_station"] = df["police_station"].fillna("UNKNOWN").str.strip()
    df["location"] = df["location"].fillna("UNKNOWN")
    df["data_sent_to_scita"] = df["data_sent_to_scita"].astype(bool)
    df["validation_status"] = df["validation_status"].fillna("not_validated")
    
    # 5. Coordinate Bounding Box Check
    df["coord_in_bengaluru_bbox"] = (
        df["latitude"].between(12.6, 13.3) & df["longitude"].between(77.2, 78.0)
    )
    
    # 6. Time Features
    cdt = df["created_datetime"]
    df["cdt_minute_of_day"] = (cdt.dt.hour * 60) + cdt.dt.minute
    df["cdt_hour"] = cdt.dt.hour
    df["cdt_day_of_week"] = cdt.dt.day_name()
    df["cdt_date"] = cdt.dt.date
    df["cdt_month"] = cdt.dt.month
    df["cdt_is_weekend"] = cdt.dt.dayofweek >= 5
    df["cdt_time_bucket"] = pd.cut(
        df["cdt_hour"],
        bins=[-1, 5, 9, 12, 16, 19, 22, 24],
        labels=["late_night", "morning_peak", "late_morning",
                "afternoon", "evening_peak", "night", "late_night2"],
    )
    
    # 7. Deduplicate
    df = df.drop_duplicates(subset="id", keep="first")
    df = df.dropna(subset=["latitude", "longitude"])
    return df

In [ ]:
def build_exploded(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    cols_keep = [
        "id", "latitude", "longitude", "location", "vehicle_type",
        "vehicle_footprint_weight", "junction_name", "has_named_junction",
        "police_station", "created_datetime", "cdt_minute_of_day", "cdt_hour", 
        "cdt_day_of_week", "cdt_date", "cdt_month", "cdt_is_weekend", 
        "cdt_time_bucket", "coord_in_bengaluru_bbox",
    ]
    base = df[cols_keep + ["violation_type_list", "offence_code_list"]]
    for row in base.itertuples(index=False):
        vt_list = row.violation_type_list
        oc_list = row.offence_code_list
        for vt, oc in zip(vt_list, oc_list):
            rec = {c: getattr(row, c) for c in cols_keep}
            rec["violation_type"] = vt
            rec["offence_code"] = oc
            rec["is_parking_violation"] = vt in PARKING_TYPES
            rows.append(rec)
    return pd.DataFrame(rows)

In [ ]:
print("Processing active dataframe pipeline...")
df_clean = clean_pipeline(df)
print(f"Cleaned event records: {df_clean.shape[0]}")

In [ ]:
df_exploded = build_exploded(df_clean)
print(f"Exploded records generated: {df_exploded.shape[0]}")

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
df_out = df_clean.drop(columns=["violation_type", "offence_code"])

df_out.to_csv(f"{OUT_DIR}/parking_violations_clean.csv", index=False)
df_exploded.to_csv(f"{OUT_DIR}/parking_violations_exploded.csv", index=False)